# HW3 — Two retrievers and a model: what RAG actually sends

**Name: Aigerim Bissembayeva**
**Student ID: S23068855**
**Group: 10**
**Repository: https://github.com/bisembaevaaigerim/css4007-hw3-bisembaevaaigerim**

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

> 

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl** = (9 + 10 + 6) / 3 = 25 / 3 = **8.3333**

**Query terms:** how, does, rag, retrieve, information (5 terms). N = 3 documents.

**TF, DF and IDF of each query term:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln(3.5 / 0.5) = ln 7 = 1.9459 |
| does | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |
| rag | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |
| information | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |


**BM25 of each document** (write out the sum for every document, one line per term that contributes):

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 1 (tie) |
| D2 | 0 | 1 (tie) |
| D3 | 0 | 1 (tie) |

All three documents tie at 0. BM25 has no ranking. It cannot choose any document. Any order is just the original order, not relevance.

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, **gener**, text, and, answer, questions | 9 |
| D2 | **retriev**, augmented, **gener**, **retriev**, relevant, documents, before, **gener**, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl** = (9 + 10 + 6) / 3 = **8.3333**

**Stemmed query terms:** how, doe, rag, retriev, inform

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |
| doe | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |
| rag | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |
| retriev | 0 | **2** | 0 | **1** | ln(2.5 / 1.5) = ln 1.6667 = **0.5108** |
| inform | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |

**BM25 of each document**, worked:

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 2 (tie) |
| D2 | 0.6847 | 1 |
| D3 | 0 | 2 (tie) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> **(a): no document.** All three score 0, so BM25 has no first document. This looks like a mistake, but it is correct. BM25 only counts exact tokens. The query words are `how, does, rag, retrieve, information`. None of them is in any document. D2 is clearly about the topic, but it says `retrieves` and `retrieval`, not `retrieve`, and it says `Retrieval-Augmented Generation`, not `RAG`. For BM25 these are different strings, so they give 0.
>
> **(b): D2 (0.6847).** After stemming, `retrieve`, `retrieves` and `retrieval` all become `retriev`. Now the query term `retriev` matches D2 two times, and D2 wins. D1 and D3 still have 0. The idf is small (0.5108), not ln 7, because now the term appears in one of the three documents.
>
> Even with stemming, BM25 still cannot see that `RAG` = `Retrieval-Augmented Generation`. An abbreviation is not a stem, so `rag` stays unmatched.

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | **0.78** | 0.9644 | 0.8124 | **0.9956** |
| Query · D2 | **1.08** | 0.9644 | 1.1225 | **0.9977** |
| Query · D3 | **0.34** | 0.9644 | 0.8307 | **0.4244** |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> **D2** (0.9977). But D1 is very close (0.9956): the difference is only 0.002. Both D1 and D2 point in almost the same direction as the query. D3 points in a different direction (big second value 0.8), so it is far behind.

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> - **A1 (a):** No. BM25 (a) gave 0 to all three, so it had no ranking. Cosine gives a full ranking: D2 > D1 > D3.
> - **A1 (b):** The **top document is the same (D2)**. But BM25 sees D1 and D3 as equal (both 0), while cosine puts D1 almost as high as D2 (0.9956) and D3 clearly last. So the order below rank 1 is different: cosine separates D1 from D3, BM25 cannot.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> BM25 compares strings. Embeddings compare meaning.
> - The query says `RAG`, D2 says `Retrieval-Augmented Generation`. Same thing, but no shared token, so BM25 gives 0. An embedding model has seen the abbreviation and the full name in similar contexts, so their vectors are close.
> - The query says `retrieve`, D2 says `retrieves` and `retrieval`. Without stemming, BM25 gives 0 (part a). With stemming it matches (part b). An embedding model treats these word forms as almost the same meaning without any stemming.
> - The query says `information`, no document has it, but D2 has `documents` and D1 has `answer questions` - related ideas. BM25 sees nothing. Embeddings can see the relation. This is also why D1 (about language models answering questions) gets a high cosine (0.9956) even though it shares no query word.
> - BM25 rewards an exact rare word strongly (idf), while embeddings can rate a document high just because the topic is close (D1), even when it does not really answer the question. So both methods can be wrong in different ways.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

The cell below only **checks** the hand calculation above. It implements the formula from README §3 directly.

In [ ]:
import math, re

docs = {
    "D1": "Large language models can generate text and answer questions.",
    "D2": "Retrieval-Augmented Generation retrieves relevant documents before generating an answer.",
    "D3": "Embeddings represent text as numerical vectors.",
}
query = "How does RAG retrieve information?"
k1, b = 1.5, 0.75

STEMS = {"retrieve": "retriev", "retrieves": "retriev", "retrieval": "retriev",
         "generate": "gener", "generating": "gener", "generation": "gener",
         "information": "inform", "does": "doe"}

def tokenize(text, stem=False):
    text = re.sub(r"[^\w\s-]", "", text.lower())        # lowercase, drop punctuation
    tokens = [t for t in re.split(r"[\s-]+", text) if t]  # split at spaces and hyphens
    return [STEMS.get(t, t) for t in tokens] if stem else tokens

def bm25_by_hand(stem):
    D = {d: tokenize(t, stem) for d, t in docs.items()}
    Q = tokenize(query, stem)
    N = len(D)
    avgdl = sum(len(t) for t in D.values()) / N
    print(f"query terms: {Q}   avgdl = {avgdl:.4f}")
    for d, toks in D.items():
        print(f"  {d} |d|={len(toks)}: {toks}")
    print(f"  {'term':<12}{'f(D1)':>6}{'f(D2)':>6}{'f(D3)':>6}{'n(t)':>6}{'idf':>9}")
    for t in Q:
        n = sum(t in toks for toks in D.values())
        idf = math.log((N - n + 0.5) / (n + 0.5))
        print(f"  {t:<12}" + "".join(f"{D[d].count(t):>6}" for d in D) + f"{n:>6}{idf:>9.4f}")
    scores = {}
    for d, toks in D.items():
        s = 0.0
        for t in Q:
            f = toks.count(t)
            n = sum(t in x for x in D.values())
            idf = math.log((N - n + 0.5) / (n + 0.5))
            s += idf * f * (k1 + 1) / (f + k1 * (1 - b + b * len(toks) / avgdl))
        scores[d] = s
    for r, (d, s) in enumerate(sorted(scores.items(), key=lambda x: -x[1]), 1):
        print(f"  rank {r}: {d}  BM25 = {s:.4f}")

print("A1 (a) — as written"); bm25_by_hand(stem=False)
print("\nA1 (b) — with stems");  bm25_by_hand(stem=True)

print("\nA2 — cosine similarity")
q = [0.8, 0.2, 0.5]
vecs = {"D1": [0.7, 0.1, 0.4], "D2": [0.9, 0.3, 0.6], "D3": [0.1, 0.8, 0.2]}
norm = lambda v: math.sqrt(sum(x * x for x in v))
cos = {}
for d, v in vecs.items():
    dot = sum(a * c for a, c in zip(q, v))
    cos[d] = dot / (norm(q) * norm(v))
    print(f"  {d}: dot = {dot:.4f}  |Q| = {norm(q):.4f}  |D| = {norm(v):.4f}  cos = {cos[d]:.4f}")
for r, (d, s) in enumerate(sorted(cos.items(), key=lambda x: -x[1]), 1):
    print(f"  rank {r}: {d}  cos = {s:.4f}")


A1 (a) — as written
query terms: ['how', 'does', 'rag', 'retrieve', 'information']   avgdl = 8.3333
  D1 |d|=9: ['large', 'language', 'models', 'can', 'generate', 'text', 'and', 'answer', 'questions']
  D2 |d|=10: ['retrieval', 'augmented', 'generation', 'retrieves', 'relevant', 'documents', 'before', 'generating', 'an', 'answer']
  D3 |d|=6: ['embeddings', 'represent', 'text', 'as', 'numerical', 'vectors']
  term         f(D1) f(D2) f(D3)  n(t)      idf
  how              0     0     0     0   1.9459
  does             0     0     0     0   1.9459
  rag              0     0     0     0   1.9459
  retrieve         0     0     0     0   1.9459
  information      0     0     0     0   1.9459
  rank 1: D1  BM25 = 0.0000
  rank 2: D2  BM25 = 0.0000
  rank 3: D3  BM25 = 0.0000

A1 (b) — with stems
query terms: ['how', 'doe', 'rag', 'retriev', 'inform']   avgdl = 8.3333
  D1 |d|=9: ['large', 'language', 'models', 'can', 'gener', 'text', 'and', 'answer', 'questions']
  D2 |d|=10: ['retriev', 

---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [2]:
import json, os, re
import numpy as np

DATA = "data/"

with open(DATA + "corpus.jsonl", encoding="utf-8") as f:
    corpus = [json.loads(line) for line in f]

with open(DATA + "queries.json", encoding="utf-8") as f:
    queries = json.load(f)["queries"] 

ids    = [c["id"] for c in corpus]
texts  = [c["text"] for c in corpus]
titles = [c["title"] for c in corpus]

print(f"{len(corpus)} chunks loaded")
print("languages:", {lang: sum(c['lang'] == lang for c in corpus) for lang in sorted({c['lang'] for c in corpus})})
print(f"{len(queries)} fixed queries:")
for q in queries:
    print(f"  {q['id']}  {q['text']}   relevant={q['relevant']}")

74 chunks loaded
languages: {'en': 71, 'kk': 1, 'ru': 2}
8 fixed queries:
  Q-01  What is form GR-305 used for?   relevant=['GO-08', 'GO-12', 'GO-23']
  Q-02  My family is poor. Is there financial help for students like me?   relevant=['GO-01', 'GO-02', 'GO-03']
  Q-03  Өтінішті қашанға дейін тапсыру керек?   relevant=['GO-09', 'GO-14']
  Q-04  What is the minimum GPA for the grant?   relevant=['GO-02']
  Q-05  When was Astana renamed Nur-Sultan, and why?   relevant=['KZ-05']
  Q-06  Where did the Soviet Union test its nuclear weapons?   relevant=['KZ-37', 'KZ-38']
  Q-07  Can the grant be sent to my mother's card?   relevant=['GO-15']
  Q-08  If my grades drop in the autumn, do I still get paid in February?   relevant=['GO-17']


## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)

>

> - **Lowercase:** yes, everything.
> - **Regex:** `\w+` — a token is a run of letters, digits or `_`. In Python 3, `\w` also matches Cyrillic and Kazakh letters, so the Russian and Kazakh chunks get real tokens. Punctuation and hyphens split tokens, so `GR-305` becomes two tokens: `gr` and `305`.
> - **Stemming:** no.
> - **Stop words:** not removed. Words like `the`, `is`, `for` stay. They appear in many chunks, so their idf is low and they add little.
> - **Parameters:** `BM25Okapi(k1=1.5, b=0.75)`, the same as Part A.

In [6]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    """lowercase + \\w+ (works for English, Russian and Kazakh); no stemming, no stop-word removal"""
    return re.findall(r"\w+", text.lower())

bm25 = BM25Okapi([tokenize(t) for t in texts], k1=1.5, b=0.75)

def bm25_search(query, k=3):
    """return the top-k chunks as a list of (index, score)"""
    scores = bm25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in top]

def show(results):
    for rank, (i, score) in enumerate(results, 1):
        first_line = texts[i][:70].replace("\n", " ")
        print(f"{rank}. {ids[i]:<6} {score:8.4f}  [{titles[i]}] {first_line}...")

print("Q-01 tokens:", tokenize(queries[0]["text"]))
show(bm25_search(queries[0]["text"]))

Q-01 tokens: ['what', 'is', 'form', 'gr', '305', 'used', 'for']
1. GO-08   15.8255  [Form GR-305: appeals] Form GR-305 is used to appeal a decision of the Grant Office. It is th...
2. GO-12    8.1953  [How to appeal a decision] A student who disagrees with a decision may appeal within 14 calendar ...
3. GO-06    8.0700  [Form GR-104: the application form] Form GR-104 is the application form for the need-based study grant. It...


## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [7]:
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

model = SentenceTransformer("google/embeddinggemma-300m", token=HF_TOKEN)

doc_emb = model.encode_document(texts, show_progress_bar=True)
print("document embeddings:", doc_emb.shape)  

def emb_search(query, k=3):
    """embed the query with the query-side prompt, cosine with every chunk, return top-k (index, score)"""
    q_emb = model.encode_query(query)
    scores = model.similarity(q_emb, doc_emb)[0].cpu().numpy()  
    top = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in top]

show(emb_search(queries[0]["text"]))

Batches: 100%|██████████| 3/3 [00:36<00:00, 12.24s/it]

document embeddings: (74, 768)
1. GO-08    0.6955  [Form GR-305: appeals] Form GR-305 is used to appeal a decision of the Grant Office. It is th...
2. GO-12    0.4609  [How to appeal a decision] A student who disagrees with a decision may appeal within 14 calendar ...
3. GO-23    0.4505  [Апелляция: кратко] Если студент не согласен с решением, он может подать апелляцию по форм...


## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | Which form is GR-210? | GO-07, GO-03 | **BM25.** The query has the exact code `210`. Only GO-07, GO-03, GO-10 and GO-13 contain it, and it is rare, so its idf is high. The handbook has three forms with almost the same names (GR-104, GR-210, GR-305). For an embedding model they look very similar, so it may blur them and return the wrong form. |
| Q-10 | Can my teachers see how much my parents earn? | GO-22 | **Embeddings.** This is a paraphrase in the same language. GO-22 says "The Grant Office does not share income information with lecturers". The query uses other words for the same ideas: teachers - lecturers, how much my parents earn - income information, see - share. BM25 has almost no useful shared word with GO-22, so it should miss it. EmbeddingGemma compares meaning, so it should find GO-22. |

In [3]:
my_queries = [
    {"id": "Q-09", "text": "Which form is GR-210?",
     "relevant": ["GO-07", "GO-03"]},
    {"id": "Q-10", "text": "Can my teachers see how much my parents earn?",
     "relevant": ["GO-22"]},
]
all_queries = queries + my_queries
print(len(all_queries), "queries in total")

10 queries in total


## Results for all ten or more queries: top 3 from each retriever

In [8]:
def hit_at_3(results, relevant):
    return any(ids[i] in relevant for i, _ in results[:3])

results = {}   
for q in all_queries:
    b = bm25_search(q["text"])
    e = emb_search(q["text"])
    results[q["id"]] = {"bm25": b, "emb": e,
                        "bm25_hit": hit_at_3(b, q["relevant"]),
                        "emb_hit": hit_at_3(e, q["relevant"])}
    print("=" * 100)
    print(f"{q['id']}: {q['text']}")
    print(f"   relevant: {q['relevant']}")
    print(f" BM25            hit@3 = {results[q['id']]['bm25_hit']}")
    show(b)
    print(f" EmbeddingGemma  hit@3 = {results[q['id']]['emb_hit']}")
    show(e)

mark = lambda x: "✓" if x else "✗"
print("\n| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |")
print("|---|---|:---:|---|:---:|")
for q in all_queries:
    r = results[q["id"]]
    print(f"| {q['id']} | {', '.join(ids[i] for i, _ in r['bm25'])} | {mark(r['bm25_hit'])} "
          f"| {', '.join(ids[i] for i, _ in r['emb'])} | {mark(r['emb_hit'])} |")
n = len(all_queries)
print(f"| **hit@3 total** | | {sum(r['bm25_hit'] for r in results.values())}/{n} "
      f"| | {sum(r['emb_hit'] for r in results.values())}/{n} |")

Q-01: What is form GR-305 used for?
   relevant: ['GO-08', 'GO-12', 'GO-23']
 BM25            hit@3 = True
1. GO-08   15.8255  [Form GR-305: appeals] Form GR-305 is used to appeal a decision of the Grant Office. It is th...
2. GO-12    8.1953  [How to appeal a decision] A student who disagrees with a decision may appeal within 14 calendar ...
3. GO-06    8.0700  [Form GR-104: the application form] Form GR-104 is the application form for the need-based study grant. It...
 EmbeddingGemma  hit@3 = True
1. GO-08    0.6955  [Form GR-305: appeals] Form GR-305 is used to appeal a decision of the Grant Office. It is th...
2. GO-12    0.4609  [How to appeal a decision] A student who disagrees with a decision may appeal within 14 calendar ...
3. GO-23    0.4505  [Апелляция: кратко] Если студент не согласен с решением, он может подать апелляцию по форм...
Q-02: My family is poor. Is there financial help for students like me?
   relevant: ['GO-01', 'GO-02', 'GO-03']
 BM25            hit@3 = False


## B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | ✓ | GO-08, GO-12, GO-23 | ✓ |
| Q-02 | KZ-17, GO-20, KZ-25 | ✗ | GO-01, GO-10, GO-20 | ✓ |
| Q-03 | GO-14, KZ-01, KZ-03 | ✓ | GO-14, GO-09, GO-11 | ✓ |
| Q-04 | GO-05, GO-17, GO-19 | ✗ | GO-02, GO-05, GO-17 | ✓ |
| Q-05 | KZ-05, KZ-06, KZ-11 | ✓ | KZ-05, KZ-06, KZ-03 | ✓ |
| Q-06 | KZ-37, KZ-38, KZ-03 | ✓ | KZ-37, KZ-38, KZ-10 | ✓ |
| Q-07 | GO-21, GO-16, GO-01 | ✗ | GO-15, GO-18, GO-01 | ✓ |
| Q-08 | GO-17, GO-05, GO-04 | ✓ | GO-17, GO-18, GO-15 | ✓ |
| Q-09 | GO-07, GO-03, GO-08 | ✓ | GO-07, GO-10, GO-08 | ✓ |
| Q-10 | GO-13, GO-21, GO-09 | ✗ | GO-10, GO-22, GO-13 | ✓ |
| **hit@3 total** | | 6/10 | | 10/10 |

**Both retrievers agree:** which query, and why was it easy?

> **Q-06** ("Where did the Soviet Union test its nuclear weapons?"). Both retrievers put **KZ-37** and **KZ-38** (Semipalatinsk Test Site) at rank 1 and 2. It was easy because the question uses the **same words as the chunk**: KZ-38 says "The Soviet Union conducted 456 nuclear tests". So BM25 matched `soviet`, `union`, `test`, `nuclear`, `weapons` and gave high scores (19.09 and 17.17, while the 3rd chunk has only 9.35). For EmbeddingGemma the meaning is also clear: one English Wikipedia topic and no trap chunk (0.596 and 0.510, the 3rd chunk only 0.332). When the query and the answer share both the words and the meaning, both methods work.

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

> **Q-09** ("Which form is GR-210?"). Both have hit@3, so my prediction was only half right, but BM25 was more precise. BM25 returned **both** relevant chunks: GO-07 (11.67) and GO-03 (8.99). The exact string it matched is **`210`** (my tokenizer splits `GR-210` into `gr` and `210`). This number appears in very few chunks, so it is a strong signal for BM25. EmbeddingGemma returned only one relevant chunk: GO-07 (0.603), then GO-10 (0.472) and **GO-08 (0.425), which is form GR-305, the appeals form**. For the embedding model, all "Form GR-xxx" chunks mean almost the same thing ("a form of the Grant Office"), so it blurred the three similar form names. For BM25, `210` is a separate word, so the number decides the ranking.
>
> I found no query where BM25 had hit@3 and EmbeddingGemma did not. BM25's advantage is precision on an exact code, not finding more answers.

**Embeddings did better:** which query, and what could BM25 not match?

> **Q-07** ("Can the grant be sent to my mother's card?"). The answer is only in **GO-15**, which is in **Russian**: «перевод на счёт родителей или других лиц невозможен» (a transfer to the parents' account is not possible). BM25 could not match **another language**: GO-15 has no word in common with the English query, so it is not in BM25's top 3. BM25's first result was **GO-21** ("Contacting the Grant Office", 14.65). It matched only general words like `can`, `the`, `grant`, `be`, `sent`, `to`, and in GO-21 "sent" means *questions can be sent through the portal*, not money. EmbeddingGemma is multilingual, so "mother's card" is close in meaning to "счёт родителей", and it put **GO-15 first** (0.422). But the margin is small (GO-18 has 0.417).
>
> The same pattern appears in other queries:
> - **Q-10** (my own, a **paraphrase** in the same language): GO-22 says "The Grant Office does not share income information with **lecturers**". The query uses other words: teachers - lecturers, how much my parents earn - income information, see - share. BM25 missed GO-22 completely. Its first result was **GO-13** ("Changes in circumstances", 8.98), which is about reporting a change of household income, not about who can see it. EmbeddingGemma found GO-22, as I predicted, **but only at rank 2** (0.364). Its first result was **GO-10** "What counts as proof of income" (0.397). It understood the topic "parents' income", but the key idea of the question ("who can see it") was weaker in the vector. So my prediction was right, but the embedding model also did not fully understand the question.
> - **Q-04** (minimum GPA): BM25 put **GO-05** first. Its title is "**Archived: the 2025 scheme (no longer in force)**". BM25 matched the exact words `minimum` and `gpa`. The correct chunk GO-02 says "cumulative GPA is **at least** 2.67", without the word `minimum`, so BM25 missed it. EmbeddingGemma understood that "at least" ≈ "minimum" and put GO-02 first (0.665), **but GO-05 is very close (0.621)**. Neither method understands that "archived" means "not valid any more".
> - **Q-02** (paraphrase): the query says "poor" and "financial help", the handbook says "low-income household" and "grant". BM25's first result was **KZ-17 (Lake Balkhash)**, because it matched general words like `like` ("like the Aral Sea"), `there` and `is`, nothing about money. EmbeddingGemma matched the meaning and put GO-01 first.
> - **Q-03** (Kazakh): BM25 found GO-14 (10.70), but every other chunk scored **0.0000**, so ranks 2 and 3 (KZ-01, KZ-03) are just a tie in corpus order, like Part A (a). BM25 could not find the English chunk GO-09. EmbeddingGemma found both GO-14 and GO-09.
>
> **Total hit@3: BM25 6/10, EmbeddingGemma 10/10.** BM25 wins only when the query contains an exact rare string (a code like `210` or `305`). Embeddings win on synonyms, paraphrases and other languages, and this corpus has many of them. But embeddings are not perfect: in Q-04, Q-07 and Q-10 the correct chunk was only a little above a wrong one.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [11]:
import tiktoken
from openai import OpenAI

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
    try:
        from google.colab import userdata
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    except ImportError:
        pass
print("OpenAI key found:", OPENAI_API_KEY is not None)

client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-5.6-luna"                      

enc = tiktoken.get_encoding("o200k_base")
def n_tokens(s):
    return len(enc.encode(s))

with open(DATA + "qa.json", encoding="utf-8") as f:
    qa = json.load(f)["questions"]
for q in qa:
    print(f"{q['id']}: {q['question']}   relevant={q['relevant']}")

SYSTEM = """Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3]."""

def build_context(found):
    """numbered context blocks: [n] (chunk id) chunk text"""
    return "\n\n".join(f"[{n}] ({ids[i]}) {texts[i]}" for n, (i, _) in enumerate(found, 1))

def build_user(question, found):
    return f"Context:\n\n{build_context(found)}\n\nQuestion: {question}"

def rag_messages(question, found):
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": build_user(question, found)}]

def ask(messages):
    """send to the model, return (answer text, usage)"""
    r = client.chat.completions.create(model=MODEL, messages=messages)
    return r.choices[0].message.content, r.usage

OpenAI key found: True
C-01: How much does a student in income band 2 receive, and when is it paid?   relevant=['GO-04', 'GO-15']
C-02: My grant application was refused. How long do I have to appeal, and which form do I use?   relevant=['GO-08', 'GO-12', 'GO-23']
C-03: What is the minimum GPA needed to qualify for the need-based grant?   relevant=['GO-02']
C-04: When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?   relevant=['KZ-43']
C-05: When was Narxoz University founded, and what was it called at the start?   relevant=['KZ-44']
C-06: I am a master's student. How much need-based grant can I get?   relevant=['GO-20']


## C1 — the complete prompt, printed before it is sent

In [12]:
c1 = next(q for q in qa if q["id"] == "C-02")
found = emb_search(c1["question"], k=3)     
context = build_context(found)
user_msg = build_user(c1["question"], found)
messages = rag_messages(c1["question"], found)

print("=" * 40, "SYSTEM MESSAGE", "=" * 40)
print(SYSTEM)
print("=" * 40, "USER MESSAGE", "=" * 40)
print(user_msg)
print("=" * 95)

t_sys, t_q, t_ctx, t_user = n_tokens(SYSTEM), n_tokens(c1["question"]), n_tokens(context), n_tokens(user_msg)
t_wrap = t_user - t_q - t_ctx                     
t_total = t_sys + t_user

answer, usage = ask(messages)

print(f"\n{'Part of the input':<40}{'Tokens':>8}")
print(f"{'System instructions':<40}{t_sys:>8}")
print(f"{'Question':<40}{t_q:>8}")
print(f"{'Retrieved context':<40}{t_ctx:>8}")
print(f"{'Template words (Context:/Question:)':<40}{t_wrap:>8}")
print(f"{'Total input (tiktoken)':<40}{t_total:>8}")
print(f"{'usage.prompt_tokens (API)':<40}{usage.prompt_tokens:>8}")
print(f"{'Difference':<40}{usage.prompt_tokens - t_total:>8}")

print(f"\nNumber of retrieved chunks: {len(found)}")
for i, _ in found:
    print(f"  {ids[i]}: {n_tokens(texts[i])} tokens, {len(texts[i])} characters")
print(f"Total context size: {t_ctx} tokens / {len(context)} characters")

print("\nMODEL ANSWER:\n" + answer)

======================================== SYSTEM MESSAGE ========================================
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].
======================================== USER MESSAGE ========================================
Context:

[1] (GO-08) Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.

[2] (GO-12) A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-.... You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | |
| Question | |
| Retrieved context | |
| **Total input** | |
| **`usage.prompt_tokens` from the response** | |

- Number of retrieved chunks:
- Tokens per chunk:
- Total context size (tokens / characters):

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

>

**What exactly does retrieval add to the model's input?**

>

## C2 — three systems, six questions

Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | | | |
| C-02 | | | |
| C-03 | | | |
| C-04 | | | |
| C-05 | | | |
| C-06 | | | |
| **agrees with `reference`** | /6 | /6 | /6 |

## C3 — question 1 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>

**3. What was sent to the model** (the context blocks):

>

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>

## C3 — question 2 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>

**3. What was sent to the model** (the context blocks):

>

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>

## C3 — question 3 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>

**3. What was sent to the model** (the context blocks):

>

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

>

**2. When can BM25 perform better than embeddings?**

>

**3. When can embeddings perform better than BM25?**

>

**4. Why is cosine similarity used with embeddings?**

>

**5. What information is actually sent to an LLM in a RAG system?**

>

**6. Does the LLM search the original documents itself?**

>

**7. What happens if retrieval returns an irrelevant chunk?**

>

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

>

**9. Is a vector database required to build a RAG system?**

>

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

>

---
# Conclusion

>